# Notebook 2: Model Architectures

**What:** Demucs, HDemucs, and HTDemucs — structure, parameters, input/output shapes.

**Why:** Choosing and modifying architectures requires understanding their building blocks (U-Net encoder-decoder, hybrid branches, Transformer).

**How:** Instantiate models with small configs, run forward passes, compare sizes.

## 1. Common Setup

All models expect: `(batch, channels, samples)` and return `(batch, sources, channels, samples)`.

In [ ]:
import sys
sys.path.insert(0, r'D:\demucs')

import torch
from demucs.demucs import Demucs
from demucs.hdemucs import HDemucs
from demucs.htdemucs import HTDemucs

sources = ['drums', 'bass', 'other', 'vocals']
sr = 44100
segment_sec = 4  # Short for memory (3070 Ti)
batch = 2
x = torch.randn(batch, 2, sr * segment_sec)  # (B, C, T)
print(f"Input shape: {x.shape}")

## 2. Demucs (Waveform U-Net)

**What:** Encoder-decoder over waveform. Strided conv downsamples, transposed conv upsamples. Optional LSTM, DConv residual branches.

**Why waveform:** No STFT phase issues; end-to-end learning.

Use smaller `channels` and `depth` for 3070 Ti.

In [ ]:
model_demucs = Demucs(
    sources=sources,
    audio_channels=2,
    channels=32,      # default 64; reduce for memory
    depth=4,          # default 6
    kernel_size=8,
    stride=4,
    segment=segment_sec,
)

n_params = sum(p.numel() for p in model_demucs.parameters())
print(f"Demucs params: {n_params / 1e6:.2f}M")

with torch.no_grad():
    out = model_demucs(x)
print(f"Output shape: {out.shape}  # (batch, sources, channels, samples)")

## 3. HDemucs (Hybrid)

**What:** Two parallel branches — time (waveform) and frequency (spectrogram). They merge at bottleneck, then split again in decoder.

**Why hybrid:** Frequency branch captures harmonics; time branch preserves phase.

In [ ]:
model_hdemucs = HDemucs(
    sources=sources,
    audio_channels=2,
    channels=32,
    depth=4,
    segment=segment_sec,
)

n_params = sum(p.numel() for p in model_hdemucs.parameters())
print(f"HDemucs params: {n_params / 1e6:.2f}M")

with torch.no_grad():
    out = model_hdemucs(x)
print(f"Output shape: {out.shape}")

## 4. HTDemucs (Hybrid + Transformer)

**What:** Same hybrid structure, but encoder outputs feed a CrossTransformer before decoder. Self-attention + cross-attention across time/freq.

**Why Transformer:** Better long-range modeling than pure convolution.

**Note:** HTDemucs has `segment` built-in; max ~7.8s for default config.

In [ ]:
model_htdemucs = HTDemucs(
    sources=sources,
    audio_channels=2,
    channels=32,
    depth=4,
    t_layers=3,       # default 5; reduce for memory
    t_heads=4,
    segment=segment_sec,
)

n_params = sum(p.numel() for p in model_htdemucs.parameters())
print(f"HTDemucs params: {n_params / 1e6:.2f}M")

with torch.no_grad():
    out = model_htdemucs(x)
print(f"Output shape: {out.shape}")

## 5. Parameter Comparison

Small configs for 3070 Ti. Full HTDemucs (channels=48, t_layers=5) is larger.

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

print("Model          | Params (M)")
print("---------------+------------")
print(f"Demucs (32ch)  | {count_params(model_demucs)/1e6:.2f}")
print(f"HDemucs (32ch) | {count_params(model_hdemucs)/1e6:.2f}")
print(f"HTDemucs (32ch)| {count_params(model_htdemucs)/1e6:.2f}")

## 6. Memory Check on GPU

Run one forward pass and check VRAM usage.

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    x_gpu = x.to(device)
    model_htdemucs.to(device)
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        _ = model_htdemucs(x_gpu)
    mb = torch.cuda.max_memory_allocated() / 1e6
    print(f"HTDemucs forward pass: ~{mb:.0f} MB peak VRAM")
else:
    print("CUDA not available; run on CPU only")

**Next:** Notebook 3 — Inference pipeline (apply_model, chunking, separation).